# Exploration des données traitées (Processed Data)

Ce notebook a pour but d'explorer les fichiers générés après les étapes de nettoyage (clean), d'ingénierie des caractéristiques (feature engineering) et de construction de la cible (target building). Nous allons visualiser les colonnes et quelques échantillons des fichiers suivants :
1. `main_table.csv`
2. `feature_matrix.csv`
3. `training_set.csv`

## 1. Main Table (`main_table.csv`)

Il s'agit de la table principale fusionnant les lignes de commandes, les informations sur les factures et les données de localisation (GPS) des clients. C'est la donnée de base, propre, avant toute agrégation complexe ou calcul de features ML.

In [ ]:
import pandas as pd

main_table_path = "../data/processed/main_table.csv"
df_main = pd.read_csv(main_table_path, parse_dates=["date_commande"])

print(f"Shape: {df_main.shape}")
print(f"Colonnes: {list(df_main.columns)}\n")
display(df_main.sample(5))

## 2. Feature Matrix (`feature_matrix.csv`)

Ce fichier contient les caractéristiques (features) construites pour le Machine Learning. Le script de Feature Engineering a agrégé les historiques d'achats pour chaque paire (Client, Produit) afin de créer des indicateurs prédictifs divisés en 5 groupes :
- **Historique de commandes** (fréquence, quantité moyenne, récence, tendance...)
- **Saisonnalité** (meilleur mois, coefficient saisonnier...)
- **Géographie** (latitude, longitude, présence de GPS...)
- **Produit** (catégorie, nouveauté, produit en vrac...)
- **Profil Client** (nombre total de factures, de produits achetés, panier moyen...)

In [ ]:
feature_matrix_path = "../data/processed/feature_matrix.csv"
df_features = pd.read_csv(feature_matrix_path)

print(f"Shape: {df_features.shape}")
print(f"Colonnes: {list(df_features.columns)}\n")
display(df_features.sample(5))

## 3. Training Set (`training_set.csv`) & Explication du "Negative Sampling"

Le **Training Set** est le jeu de données final fourni au modèle (ex: XGBoost). Il combine la `feature_matrix` avec la **cible (target)** à prédire (`target_bought` : le client a-t-il acheté ce produit le mois suivant ?).

### Qu'est-ce que le Negative Sampling (Échantillonnage Négatif) ?
Dans la vraie vie, un client n'achète qu'une infime fraction du catalogue disponible. Si l'on créait une ligne pour **toutes** les combinaisons possibles de (Client, Produit), on obtiendrait un dataset gigantesque rempli presque exclusivement de `target_bought = 0` (non acheté). Le modèle serait déséquilibré, la mémoire de la machine exploserait, et le modèle mettrait trop de temps à s'entraîner sur des cas évidents.

**La solution (implémentée dans `target_builder.py`) :**
1. **Pairs Positifs (Positive pairs) :** On garde toutes les fois où un client a effectivement commandé un produit. L'ancre est fixée au dernier mois d'achat, et on regarde le mois d'après pour voir s'il y a eu réachat (`target_bought = 1` ou `0`).
2. **Pairs Négatifs (Negative Sampling) :** Pour chaque client, on choisit aléatoirement un petit nombre de produits qu'il n'a **jamais** commandés. Dans notre cas, le ratio défini est de `3:1` (3 exemples négatifs pour 1 positif). Pour ces paires artificielles, la cible est forcément `target_bought = 0`.

Cela permet d'apprendre au modèle à quoi ressemble un "non-achat" de manière très efficace, sans exploser la taille de nos données !

In [ ]:
training_set_path = "../data/processed/training_set.csv"
df_train = pd.read_csv(training_set_path)

print(f"Shape: {df_train.shape}")
print(f"Colonnes: {list(df_train.columns)}\n")

print("Distribution de la variable cible (target_bought):")
display(df_train["target_bought"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

print("\nExemples de lignes:")
display(df_train.sample(5))